In [16]:
import os
import pandas as pd
import numpy as np


def load_and_combine_csvs(files, period_name=None):
    """
    Load multiple CSV files and combine them into one DataFrame.

    Parameters
    ----------
    files : list of str
        CSV file paths.
    period_name : str, optional
        Label for the period, added as a column.

    Returns
    -------
    pd.DataFrame
    """
    dfs = []

    for file in files:
        df = pd.read_csv(file, low_memory=False, encoding="latin1")

        filename = os.path.basename(file)
        year = filename[:4]

        df["year"] = pd.to_numeric(year, errors="coerce")

        if period_name is not None:
            df["period"] = period_name

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def clean_flight_data(df):
    """
    Clean flight dataset and convert relevant columns to numeric.
    """
    df = df.copy()
    df = df.replace("NA", np.nan)

    numeric_cols = [
        "AirTime",
        "ActualElapsedTime",
        "CRSElapsedTime",
        "ArrDelay",
        "DepDelay",
        "Distance",
        "Cancelled",
        "Diverted",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    if "Distance" in df.columns:
        df["Distance_km"] = df["Distance"] * 1.60934

    df = df.drop(columns=["Distance"])
    df = df.rename(columns={"Distance_km": "Distance"})
        
    return df


def build_airport_nodes(df):
    """
    Create airport-level node analysis table.

    Each row is one airport (IATA code), with counts and average metrics.
    """
    required_cols = ["Origin", "Dest"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    origin_stats = (
        df.groupby("Origin")
        .agg(
            origin_flight_count=("Origin", "size"),
            avg_actual_elapsed_time=("ActualElapsedTime", "mean"),
            avg_scheduled_elapsed_time=("CRSElapsedTime", "mean"),
            avg_air_time=("AirTime", "mean"),
            avg_distance=("Distance", "mean"),
            avg_dep_delay=("DepDelay", "mean"),
            cancelled_origin_count=("Cancelled", "sum"),
            diverted_origin_count=("Diverted", "sum"),
            unique_destinations=("Dest", "nunique"),
        )
        .reset_index()
        .rename(columns={"Origin": "IATA"})
    )

    dest_stats = (
        df.groupby("Dest")
        .agg(
            dest_flight_count=("Dest", "size"),
            avg_arr_delay=("ArrDelay", "mean"),
            unique_origins=("Origin", "nunique"),
        )
        .reset_index()
        .rename(columns={"Dest": "IATA"})
    )

    airport_nodes = pd.merge(origin_stats, dest_stats, on="IATA", how="outer")

    count_cols = [
        "origin_flight_count",
        "dest_flight_count",
        "cancelled_origin_count",
        "diverted_origin_count",
        "unique_destinations",
        "unique_origins",
    ]

    for col in count_cols:
        airport_nodes[col] = airport_nodes[col].fillna(0)

    airport_nodes["flight_count"] = (
        airport_nodes["origin_flight_count"] + airport_nodes["dest_flight_count"]
    )

    airport_nodes["unique_connections"] = (
        airport_nodes["unique_destinations"] + airport_nodes["unique_origins"]
    )

    airport_nodes["avg_flight_time"] = airport_nodes["avg_actual_elapsed_time"]

    round_cols = [
        "avg_flight_time",
        "avg_actual_elapsed_time",
        "avg_scheduled_elapsed_time",
        "avg_air_time",
        "avg_distance",
        "avg_dep_delay",
        "avg_arr_delay",
    ]

    for col in round_cols:
        airport_nodes[col] = airport_nodes[col].round(2)

    airport_nodes = airport_nodes[
        [
            "IATA",
            "flight_count",
            "origin_flight_count",
            "dest_flight_count",
            "avg_flight_time",
            "avg_actual_elapsed_time",
            "avg_scheduled_elapsed_time",
            "avg_air_time",
            "avg_distance",
            "avg_dep_delay",
            "avg_arr_delay",
            "cancelled_origin_count",
            "diverted_origin_count",
            "unique_destinations",
            "unique_origins",
            "unique_connections",
        ]
    ].sort_values("flight_count", ascending=False)

    return airport_nodes


def save_airport_nodes_for_period(files, period_name, output_dir="."):
    """
    Run the full pipeline for one period and save a CSV.

    Parameters
    ----------
    files : list of str
        CSV files to combine for the period.
    period_name : str
        Name of the period, used in the output filename.
    output_dir : str
        Directory to save the CSV.

    Returns
    -------
    pd.DataFrame
    """
    df = load_and_combine_csvs(files, period_name=period_name)
    df = clean_flight_data(df)
    airport_nodes = build_airport_nodes(df)

    safe_period_name = period_name.replace(" ", "_").replace("-", "_")
    output_path = os.path.join(output_dir, f"airport_nodes_{safe_period_name}.csv")

    airport_nodes.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

    return airport_nodes


period_1999_2000_files = ["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/1999.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2000.csv"]
period_2002_2003_files = ["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2002.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2003.csv"]

nodes_1999_2000 = save_airport_nodes_for_period(
    period_1999_2000_files,
    "1999-2000"
)

nodes_2002_2003 = save_airport_nodes_for_period(
    period_2002_2003_files,
    "2002-2003"
)


Saved: ./airport_nodes_1999_2000.csv
Saved: ./airport_nodes_2002_2003.csv


In [15]:
nodes_1999_2000

,IATA,flight_count,origin_flight_count,dest_flight_count,avg_flight_time,avg_actual_elapsed_time,avg_scheduled_elapsed_time,avg_air_time,avg_distance,avg_dep_delay,avg_arr_delay,cancelled_origin_count,diverted_origin_count,unique_destinations,unique_origins,unique_connections
150,ORD,1190170,595022,595148,142.71,142.71,141.77,116.69,1370.14,14.56,14.24,35303,1447,98,98,196
8,ATL,1058168,529573,528595,119.88,119.88,120.38,93.49,1068.41,11.23,8.19,16137,1057,97,98,195
51,DFW,984122,491949,492173,147.93,147.93,149.56,122.35,1475.36,10.69,5.85,15963,1410,91,92,183
107,LAX,807862,403880,403982,162.65,162.65,163.38,140.57,1818.60,11.56,10.53,13715,755,64,63,127
156,PHX,733226,366629,366597,129.22,129.22,129.93,108.87,1332.34,13.22,10.38,7701,681,67,67,134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,LWB,326,163,163,73.38,73.38,71.28,60.30,583.93,5.09,5.34,4,0,3,3,6
25,BRO,296,148,148,69.04,69.04,67.03,52.42,495.68,2.88,6.62,4,2,1,1,2
130,MIB,178,89,89,78.58,78.58,85.72,65.82,735.47,-7.14,1.43,1,0,1,1,2
24,BQN,35,17,18,237.81,237.81,235.00,213.00,2550.80,-5.38,9.76,1,0,1,1,2


In [23]:
def build_edge_list(df):
    """
    Create weighted edges between airports.
    Each row = connection between two airports
    """

    edges = (
        df.groupby(["Origin", "Dest"])
        .agg(
            flight_count=("Origin", "size"),
            avg_distance=("Distance", "mean"),
            avg_air_time=("AirTime", "mean"),
            avg_dep_delay=("DepDelay", "mean"),
            avg_arr_delay=("ArrDelay", "mean"),
        )
        .reset_index()
    )
    round_cols = ["avg_distance", "avg_air_time", "avg_dep_delay", "avg_arr_delay"]

    edges[round_cols] = edges[round_cols].round(2)

    return edges

In [26]:


df_1999_2000 = load_and_combine_csvs(["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/1999.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2000.csv"], "1999-2000")
df_1999_2000 = clean_flight_data(df_1999_2000)

edges_1999_2000 = build_edge_list(df_1999_2000)
edges_1999_2000.to_csv("edges_1999_2000.csv", index=False)

df_2002_2003 = load_and_combine_csvs(["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2002.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2003.csv"], "2002_2003")
df_2002_2003 = clean_flight_data(df_2002_2003)

edges_2002_2003 = build_edge_list(df_2002_2003)
edges_2002_2003.to_csv("edges_2002_2003.csv", index=False)

In [30]:
len(edges_2002_2003) - len(edges_1999_2000)

1135

In [31]:
import pandas as pd
import networkx as nx


def build_airport_graph(nodes_df, edges_df, directed=True):
    if directed:
        G = nx.DiGraph()
    else:
        G = nx.Graph()

    # Add nodes with attributes
    for _, row in nodes_df.iterrows():
        node_id = row["IATA"]
        attrs = row.drop("IATA").to_dict()
        G.add_node(node_id, **attrs)

    # Add edges with attributes
    for _, row in edges_df.iterrows():
        source = row["Origin"]
        target = row["Dest"]
        attrs = row.drop(["Origin", "Dest"]).to_dict()
        G.add_edge(source, target, **attrs)

    return G

In [34]:
nodes_df = pd.read_csv("airport_nodes_1999_2000.csv")
edges_df = pd.read_csv("edges_1999_2000.csv")

G_before = build_airport_graph(nodes_df, edges_df, directed=True)

print("Number of nodes:", G_before.number_of_nodes())
print("Number of edges:", G_before.number_of_edges())

Number of nodes: 208
Number of edges: 3472


In [35]:
nodes_df = pd.read_csv("airport_nodes_2002_2003.csv")
edges_df = pd.read_csv("edges_2002_2003.csv")

G_after = build_airport_graph(nodes_df, edges_df, directed=True)

print("Number of nodes:", G_after.number_of_nodes())
print("Number of edges:", G_after.number_of_edges())

Number of nodes: 285
Number of edges: 4607


In [51]:
print(nx.numeric_assortativity_coefficient(G_after, "flight_count"))
print(nx.numeric_assortativity_coefficient(G_before, "flight_count"))


-0.2696023570094179
-0.2764237147499827


In [ ]:
G_before.number_of_edges()

3472

In [ ]:
density = 2 * G_before.number_of_edges() / (G_before.number_of_nodes() * (G_before.number_of_nodes() - 1))
print(density)

print(nx.density(G_before))

0.16127833519137866
0.08063916759568933


#### network visualization

In [59]:
import networkx as nx
from netwulf import visualize
import ast
import pandas as pd
import itertools as iter
from collections import Counter
import matplotlib
matplotlib.use("Agg")


In [60]:
strength_dict = dict(G_before.degree(weight="weight"))
nx.set_node_attributes(G_before, strength_dict, "strength")